GENERAR DATASET USANDO PANDAS

In [554]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("shayanzk/imdb-top-100-movies-dataset-2025-edition")
file = path + "/top_100_movies_full_best_effort.csv"

df = pd.read_csv(file)
df.drop(columns=["Rank","Title","Main Actor(s)", "Director","Country","Language","Box Office ($M)"], inplace=True)
df.head()


,Year,Genre(s),IMDb Rating,Rotten Tomatoes %,Runtime (mins),Oscars Won,Metacritic Score
0,1994,Drama,9.3,91.0,142.0,0,82.0
1,1972,Crime|Drama,9.2,98.0,175.0,3,100.0
2,2008,Action|Crime|Drama,9.0,94.0,152.0,2,84.0
3,1974,Crime|Drama,9.0,97.0,202.0,6,90.0
4,1957,Crime|Drama,9.0,100.0,96.0,0,96.0


ENCODING DE CADENAS DE TEXTO 

In [555]:
from sklearn.preprocessing import MultiLabelBinarizer

# One-hot encoding de la columna "Genre(s)"
mlb = MultiLabelBinarizer()
genres_split = df['Genre(s)'].str.split('|')
genre_matrix = mlb.fit_transform(genres_split)

# Crear DataFrame con los géneros codificados
genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_)
df = pd.concat([df.drop(columns=['Genre(s)']), genre_df], axis=1)


CONVERTIR A MATRIZ NUMPY 

In [556]:


# Excluir la columna "Oscars Won" para X, ya que será la variable target
X = df.drop(columns=["Oscars Won"]).to_numpy()

In [557]:
print(X)

[[1994.     9.3   91.  ...    0.     0.     0. ]
 [1972.     9.2   98.  ...    0.     0.     0. ]
 [2008.     9.    94.  ...    0.     0.     0. ]
 ...
 [1944.     8.    98.  ...    0.     0.     0. ]
 [1977.     8.    97.  ...    0.     0.     0. ]
 [1959.     8.2   94.  ...    0.     0.     0. ]]


In [558]:
# Muestra todos los géneros que existen en el dataset (directamente desde MultiLabelBinarizer)
genre_columns = list(mlb.classes_)


Divide a los datos en clases

In [559]:
# Si el número de oscar es mayor a 0, entra en la categoría de "ganó oscar" (1), sino, "no ganó oscar" (0)
Y = (df["Oscars Won"] > 0).astype(int)

CLASIFICAR EN UN ARBOL DE DECISIÓN

In [560]:
from sklearn import tree

clf = tree.DecisionTreeClassifier(
    max_depth=3,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

clf = clf.fit(X, Y)



PREDICE SI UNA PELICULA GANA EL OSCAR EN FUNCIÓN DE LOS VALORES QUE INGRESEMOS

In [561]:
# Función helper para hacer predicciones 
"""
    Predice si una película gana Oscar basada en sus características.
    
"""
def predict_oscar(year, rating, rotten_tomatoes, runtime, metacritic, genres_list):
    
    genre_vector = mlb.transform([genres_list])[0] #Transforma la lista de géneros en un vector codificado 
    
   
    features = [year, rating, rotten_tomatoes, runtime, metacritic] + list(genre_vector)
    
    return clf.predict([features])[0]

In [562]:
# Mulholland Drive, 2001 - Crime, Drama, Film-Noir, Mystery, Thriller
prediction = predict_oscar(
    year=2001,
    rating=7.9,
    rotten_tomatoes=83.0,
    runtime=147,
    metacritic=87,
    genres_list=['Crime', 'Drama', 'Film-Noir', 'Mystery', 'Thriller']
)
print(f"Mulholland Drive: {'Gana Oscar' if prediction == 1 else 'No gana Oscar'}")

Mulholland Drive: Gana Oscar


In [563]:
# Spider-Man, 2002 - Action, Adventure, Sci-Fi
prediction = predict_oscar(
    year=2002,
    rating=7.4,
    rotten_tomatoes=90,
    runtime=121,
    metacritic=73,
    genres_list=['Action', 'Adventure', 'Sci-Fi']
)
print(f"Spider-Man: {'Gana Oscar' if prediction == 1 else 'No gana Oscar'}")

Spider-Man: No gana Oscar


In [564]:
# Emilia Pérez, 2024 - Drama, Music, Thriller
prediction = predict_oscar(
    year=2024,
    rating=5.3,
    rotten_tomatoes=70,
    runtime=130,
    metacritic=70,
    genres_list=['Drama', 'Music', 'Thriller']
)
print(f"Emilia Pérez: {'Gana Oscar' if prediction == 1 else 'No gana Oscar'}")

Emilia Pérez: No gana Oscar


Hace predicciones solo teniendo en cuenta a los géneros de la pelicula, ignorando otros campos

In [565]:
# Entrenar un clasificador usando solo los genres
X_genres = df[genre_columns].to_numpy()
Y_genres = (df["Oscars Won"] > 0).astype(int)

clf_genres = tree.DecisionTreeClassifier()
clf_genres = clf_genres.fit(X_genres, Y_genres)

# Función helper para predicciones basadas solo en géneros
def predict_oscar_by_genres(genres_list):
    genre_vector = mlb.transform([genres_list])[0]
    return clf_genres.predict([genre_vector])[0]



In [566]:
# Mulholland Drive, 2001 - Crime, Drama, Film-Noir, Mystery, Thriller
prediction = predict_oscar_by_genres(['Crime', 'Drama', 'Film-Noir', 'Mystery', 'Thriller'])
print(f"Mulholland Drive: {'Gana Oscar' if prediction == 1 else 'No gana Oscar'}")

Mulholland Drive: Gana Oscar


In [567]:
# Spider-Man, 2002 - Action, Adventure, Sci-Fi
prediction = predict_oscar_by_genres(['Action', 'Adventure', 'Sci-Fi'])
print(f"Spider-Man: {'Gana Oscar' if prediction == 1 else 'No gana Oscar'}")

Spider-Man: No gana Oscar


In [568]:
# Emilia Pérez, 2024 - Drama, Music, Thriller
prediction = predict_oscar_by_genres(['Drama', 'Music', 'Thriller'])
print(f"Emilia Pérez: {'Gana Oscar' if prediction == 1 else 'No gana Oscar'}")

Emilia Pérez: Gana Oscar


In [569]:
# Entrenar un clasificador usando solo características de reseñas
# Obtener todas las columnas numéricas que no sean "Oscars Won" ni géneros
review_features = [col for col in df.columns if col not in genre_columns and col != "Oscars Won"]

X_reviews = df[review_features].to_numpy()
Y_reviews = (df["Oscars Won"] > 0).astype(int)

clf_reviews = tree.DecisionTreeClassifier()
clf_reviews = clf_reviews.fit(X_reviews, Y_reviews)

# Función helper para predicciones basadas solo en reseñas
def predict_oscar_by_reviews(year, rating, rotten_tomatoes, runtime, metacritic):
  
    features = [year, rating, rotten_tomatoes, runtime, metacritic]
    return clf_reviews.predict([features])[0]



In [570]:
# Mulholland Drive, 2001
prediction = predict_oscar_by_reviews(year=2001, rating=7.9, rotten_tomatoes=83.0, runtime=147, metacritic=87)
print(f"Mulholland Drive: {'Gana Oscar' if prediction == 1 else 'No gana Oscar'}")

Mulholland Drive: Gana Oscar


In [571]:
# Spider-Man, 2002
prediction = predict_oscar_by_reviews(year=2002, rating=7.4, rotten_tomatoes=90, runtime=121, metacritic=73)
print(f"Spider-Man: {'Gana Oscar' if prediction == 1 else 'No gana Oscar'}")


Spider-Man: No gana Oscar


In [572]:
# Emilia Pérez, 2024
prediction = predict_oscar_by_reviews(year=2024, rating=5.3, rotten_tomatoes=70, runtime=130, metacritic=70)
print(f"Emilia Pérez: {'Gana Oscar' if prediction == 1 else 'No gana Oscar'}")

Emilia Pérez: No gana Oscar
